In [ ]:
import os
import glob
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Markdown

# 尝试加载 Tokenizer，如果断网则回退到基础按词切分
try:
    from transformers import AutoTokenizer
    print("⏳ 正在加载 Qwen Tokenizer...")
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B", trust_remote_code=True)
    def get_token_length(text):
        return len(tok.encode(text, add_special_tokens=False))
except Exception as e:
    print(f"⚠️ Tokenizer 加载失败 ({e})，将使用基础 split 估算长度。")
    def get_token_length(text):
        return len(text.split())

# =====================================================================
# 1. 自动扫描当前目录下的日志文件
# =====================================================================
available_files = glob.glob("./optimization_histories/optimization_history*.jsonl")
if not available_files:
    available_files = ["(当前目录下未找到 optimization_history*.jsonl 文件)"]

# =====================================================================
# 2. 构建顶层 UI
# =====================================================================
print("=====================================================")
print("📊 Latent Space Optimization 终极可视化分析引擎")
print("=====================================================")

file_dropdown = widgets.Dropdown(options=available_files, description='📁 选择日志:', layout=widgets.Layout(width='60%'))
btn_load = widgets.Button(description='加载并分析', button_style='primary', icon='play')
ui_header = widgets.HBox([file_dropdown, btn_load])
main_output = widgets.Output()

display(ui_header)
display(main_output)

# =====================================================================
# 3. 核心分析与渲染逻辑
# =====================================================================
def run_analysis(file_path):
    with main_output:
        clear_output(wait=True)
        print(f"⏳ 正在加载并解析 {file_path} ...")
        
        # --- A. 数据加载与预处理 ---
        all_data = {}
        run_config = {}  # 用于存储第一行的 config 信息
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for i, line in enumerate(f):
                    record = json.loads(line)
                    
                    # 拦截第 0 行的 config 信息
                    if i == 0 and "config" in record:
                        run_config = record["config"]
                        continue
                    
                    uid = record.get('uid')
                    if not uid: continue # 容错
                    
                    sample_id = uid.split('_')[0]
                    if sample_id not in all_data:
                        all_data[sample_id] = {}
                    
                    # 预先计算文本长度
                    for s in record['steps']:
                        if 'sample_gen_text' in s.get('metrics', {}):
                            s['metrics']['gen_text_len'] = get_token_length(s['metrics']['sample_gen_text'])
                            
                    all_data[sample_id][uid] = record
            print(f"✅ 成功加载 {len(all_data)} 个独立的 Sample(问题) 数据。\n")
        except Exception as e:
            print(f"❌ 读取文件失败: {e}")
            return

        # ==========================================
        # 渲染 Config 配置看板
        # ==========================================
        if run_config:
            config_items = "".join([
                f"""
                <div style='background: #fff; padding: 8px 12px; border-radius: 6px; box-shadow: 0 1px 2px rgba(0,0,0,0.05); border-left: 3px solid #9b59b6;'>
                    <div style='font-size: 11px; color: #7f8c8d; text-transform: uppercase;'>{k}</div>
                    <div style='font-size: 14px; color: #2c3e50; font-weight: bold;'>{v}</div>
                </div>
                """ for k, v in run_config.items()
            ])
            
            config_html = f"""
            <div style="margin-bottom: 20px; padding: 15px; background: #f8f9fa; border-radius: 8px; border: 1px solid #dcdde1;">
                <h3 style="margin-top:0; margin-bottom: 15px; color: #2c3e50;">⚙️ 实验配置 (Run Configuration)</h3>
                <div style="display: grid; grid-template-columns: repeat(auto-fill, minmax(180px, 1fr)); gap: 12px;">
                    {config_items}
                </div>
            </div>
            """
            display(HTML(config_html))

        # ==========================================
        # B. 聚合计算 Summary 统计信息
        # ==========================================
        summary_stats = {
            'last_total_loss': [], 'last_gt_loss': [], 'last_lm_loss': [], 'last_kl_loss': [],
            'last_pure_acc': [], 'last_forced_acc': [], 'last_fast_acc': [],
            'opt_pure_acc': [], 'opt_forced_acc': [], 'opt_fast_acc': [],
            'last_gen_len': []
        }
        
        for sid, resps in all_data.items():
            for uid, record in resps.items():
                steps = record.get('steps', [])
                if not steps: continue
                
                last_metrics = steps[-1].get('metrics', {})
                all_pure = [s['metrics']['pure_acc'] for s in steps if 'pure_acc' in s.get('metrics', {})]
                all_forced = [s['metrics']['forced_acc'] for s in steps if 'forced_acc' in s.get('metrics', {})]
                all_fast = [s['metrics']['fast_acc'] for s in steps if 'fast_acc' in s.get('metrics', {})]
                
                if 'total_loss' in last_metrics: summary_stats['last_total_loss'].append(last_metrics['total_loss'])
                if 'gt_loss' in last_metrics: summary_stats['last_gt_loss'].append(last_metrics['gt_loss'])
                if 'lm_loss' in last_metrics: summary_stats['last_lm_loss'].append(last_metrics['lm_loss'])
                if 'kl_loss' in last_metrics: summary_stats['last_kl_loss'].append(last_metrics['kl_loss'])
                
                if all_pure:
                    summary_stats['last_pure_acc'].append(all_pure[-1])
                    summary_stats['opt_pure_acc'].append(max(all_pure))
                if all_forced:
                    summary_stats['last_forced_acc'].append(all_forced[-1])
                    summary_stats['opt_forced_acc'].append(max(all_forced))
                if all_fast:
                    summary_stats['last_fast_acc'].append(all_fast[-1])
                    summary_stats['opt_fast_acc'].append(max(all_fast))
                
                all_lens = [s['metrics']['gen_text_len'] for s in steps if 'gen_text_len' in s.get('metrics', {})]
                if all_lens:
                    summary_stats['last_gen_len'].append(all_lens[-1])

        def mean_safe(lst): return np.mean(lst) if lst else 0.0
        
        m_total = mean_safe(summary_stats['last_total_loss'])
        m_gt = mean_safe(summary_stats['last_gt_loss'])
        m_kl = mean_safe(summary_stats['last_kl_loss'])
        m_lm = mean_safe(summary_stats['last_lm_loss'])
        m_len = mean_safe(summary_stats['last_gen_len'])

        summary_html = f"""
        <div style="display: flex; gap: 20px; flex-wrap: wrap; margin-bottom: 20px;">
            <div style="flex: 1; min-width: 200px; padding: 15px; background: #fff; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); border-top: 4px solid #e74c3c;">
                <h3 style="margin-top:0; color: #2c3e50;">📉 Loss (收敛态 / Last)</h3>
                <p style="margin:5px 0;"><b>Total:</b> {m_total:.4f}</p>
                <p style="margin:5px 0;"><b>GT Loss:</b> {m_gt:.4f}</p>
                <p style="margin:5px 0;"><b>KL Loss:</b> {m_kl:.4f}</p>
                <p style="margin:5px 0;"><b>LM Loss:</b> {m_lm:.4f}</p>
            </div>
            <div style="flex: 1; min-width: 200px; padding: 15px; background: #fff; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); border-top: 4px solid #3498db;">
                <h3 style="margin-top:0; color: #2c3e50;">🎯 Accuracy (收敛态 / Last)</h3>
                <p style="margin:5px 0;"><b>Pure:</b> {mean_safe(summary_stats['last_pure_acc']):.2%}</p>
                <p style="margin:5px 0;"><b>Forced:</b> {mean_safe(summary_stats['last_forced_acc']):.2%}</p>
                <p style="margin:5px 0;"><b>Fast:</b> {mean_safe(summary_stats['last_fast_acc']):.2%}</p>
            </div>
            <div style="flex: 1; min-width: 200px; padding: 15px; background: #fff; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); border-top: 4px solid #2ecc71;">
                <h3 style="margin-top:0; color: #2c3e50;">🚀 Accuracy (峰值 / Optimal)</h3>
                <p style="margin:5px 0;"><b>Pure:</b> {mean_safe(summary_stats['opt_pure_acc']):.2%}</p>
                <p style="margin:5px 0;"><b>Forced:</b> {mean_safe(summary_stats['opt_forced_acc']):.2%}</p>
                <p style="margin:5px 0;"><b>Fast:</b> {mean_safe(summary_stats['opt_fast_acc']):.2%}</p>
            </div>
            <div style="flex: 1; min-width: 200px; padding: 15px; background: #fff; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); border-top: 4px solid #f39c12;">
                <h3 style="margin-top:0; color: #2c3e50;">📝 Gen Text (收敛态)</h3>
                <p style="margin:5px 0;"><b>Average Tokens:</b> {m_len:.1f}</p>
            </div>
        </div>
        """
        display(HTML(summary_html))

        # ==========================================
        # C. 准备图表数据 (全局聚合 & 前向填充)
        # ==========================================
        max_step = 0
        for sid, resps in all_data.items():
            for uid, record in resps.items():
                if record['steps']: max_step = max(max_step, max(s['step'] for s in record['steps']))
        
        steps_sorted = list(range(max_step + 1))
        global_agg = {step: {'total_loss': [], 'gt_loss': [], 'lm_loss': [], 'kl_loss': [], 'pure_acc': [], 'forced_acc': [], 'fast_acc': [], 'gen_text_len': []} for step in steps_sorted}

        for sid, resps in all_data.items():
            for uid, record in resps.items():
                step_to_metrics = {s['step']: s['metrics'] for s in record['steps']}
                last_known = {'total_loss': None, 'gt_loss': None, 'lm_loss': None, 'kl_loss': None, 'pure_acc': None, 'forced_acc': None, 'fast_acc': None, 'gen_text_len': None}
                
                for step_idx in steps_sorted:
                    if step_idx in step_to_metrics:
                        m = step_to_metrics[step_idx]
                        if 'total_loss' in m: last_known['total_loss'] = m['total_loss']
                        if 'gt_loss' in m: last_known['gt_loss'] = m['gt_loss']
                        if 'lm_loss' in m: last_known['lm_loss'] = m['lm_loss']
                        if 'kl_loss' in m: last_known['kl_loss'] = m['kl_loss']
                        if 'pure_acc' in m: last_known['pure_acc'] = m['pure_acc']
                        if 'forced_acc' in m: last_known['forced_acc'] = m['forced_acc']
                        if 'fast_acc' in m: last_known['fast_acc'] = m['fast_acc']
                        if 'gen_text_len' in m: last_known['gen_text_len'] = m['gen_text_len']
                    
                    if last_known['total_loss'] is not None: global_agg[step_idx]['total_loss'].append(last_known['total_loss'])
                    if last_known['gt_loss'] is not None: global_agg[step_idx]['gt_loss'].append(last_known['gt_loss'])
                    if last_known['lm_loss'] is not None: global_agg[step_idx]['lm_loss'].append(last_known['lm_loss'])
                    if last_known['kl_loss'] is not None: global_agg[step_idx]['kl_loss'].append(last_known['kl_loss'])
                    if last_known['pure_acc'] is not None: global_agg[step_idx]['pure_acc'].append(last_known['pure_acc'])
                    if last_known['forced_acc'] is not None: global_agg[step_idx]['forced_acc'].append(last_known['forced_acc'])
                    if last_known['fast_acc'] is not None: global_agg[step_idx]['fast_acc'].append(last_known['fast_acc'])
                    if last_known['gen_text_len'] is not None: global_agg[step_idx]['gen_text_len'].append(last_known['gen_text_len'])

        avg_total_loss = [np.mean(global_agg[s]['total_loss']) if global_agg[s]['total_loss'] else None for s in steps_sorted]
        avg_gt_loss = [np.mean(global_agg[s]['gt_loss']) if global_agg[s]['gt_loss'] else None for s in steps_sorted]
        avg_lm_loss = [np.mean(global_agg[s]['lm_loss']) if global_agg[s]['lm_loss'] else None for s in steps_sorted]
        avg_kl_loss = [np.mean(global_agg[s]['kl_loss']) if global_agg[s]['kl_loss'] else None for s in steps_sorted]
        avg_pure_acc = [np.mean(global_agg[s]['pure_acc']) if global_agg[s]['pure_acc'] else None for s in steps_sorted]
        avg_forced_acc = [np.mean(global_agg[s]['forced_acc']) if global_agg[s]['forced_acc'] else None for s in steps_sorted]
        avg_fast_acc = [np.mean(global_agg[s]['fast_acc']) if global_agg[s]['fast_acc'] else None for s in steps_sorted]
        avg_gen_len = [np.mean(global_agg[s]['gen_text_len']) if global_agg[s]['gen_text_len'] else None for s in steps_sorted]

        sample_stats = {}
        for sid, resps in all_data.items():
            improved_count = 0
            total_resps = len(resps)
            imp_list = []
            for uid, record in resps.items():
                accs = [s['metrics']['pure_acc'] for s in record['steps'] if 'pure_acc' in s.get('metrics', {})]
                if accs:
                    imp = max(accs) - accs[0]
                    imp_list.append(imp)
                    if imp > 0: improved_count += 1
            
            avg_imp_pct = (np.mean(imp_list) * 100) if imp_list else 0.0
            label = f"{'🌟' if avg_imp_pct >= 20 else '❌'} {sid} ({improved_count}/{total_resps}提升, 均提: +{avg_imp_pct:.1f}%)"
            sample_stats[label] = sid

        # ==========================================
        # 控件定义区
        # ==========================================
        toggle_scale = widgets.ToggleButtons(options=['对数坐标 (Log)', '线性坐标 (Linear)'], description='📉 Loss坐标轴:', button_style='warning', layout=widgets.Layout(margin='0 0 15px 0'))
        out_global_plot = widgets.Output()
        
        dropdown_sample = widgets.Dropdown(options=list(sample_stats.keys()), description='选择问题:', layout=widgets.Layout(width='60%'))
        out_problem = widgets.Output()
        out_plots = widgets.Output()

        dropdown_uid = widgets.Dropdown(description='选择变体:', layout=widgets.Layout(width='25%'))
        slider_step = widgets.SelectionSlider(description='Step:', options=[0], continuous_update=False, layout=widgets.Layout(width='50%'))
        
        # 🆕 将原本的 ToggleButtons 升级为动态的单个 ToggleButton
        btn_markdown = widgets.ToggleButton(
            value=False,
            description=' 💡 渲染 Markdown',
            button_style='info', 
            tooltip='点击在 Markdown 渲染与原始文本之间切换',
            layout=widgets.Layout(width='20%')
        )
        
        out_text = widgets.Output()

        # ==========================================
        # 绘图与更新函数
        # ==========================================
        def draw_global_plot():
            with out_global_plot:
                clear_output(wait=True)
                fig_global = make_subplots(specs=[[{"secondary_y": True}]])
                fig_global.add_trace(go.Scatter(x=steps_sorted, y=avg_total_loss, mode='lines', name='Total Loss', line=dict(color='black', width=3)), secondary_y=False)
                fig_global.add_trace(go.Scatter(x=steps_sorted, y=avg_gt_loss, mode='lines', name='GT Loss', line=dict(color='red', width=2)), secondary_y=False)
                fig_global.add_trace(go.Scatter(x=steps_sorted, y=avg_kl_loss, mode='lines', name='KL Loss', line=dict(color='blue', width=2)), secondary_y=False)
                fig_global.add_trace(go.Scatter(x=steps_sorted, y=avg_lm_loss, mode='lines', name='LM Loss', line=dict(color='orange', dash='dash')), secondary_y=False)
                
                fig_global.add_trace(go.Scatter(x=steps_sorted, y=avg_pure_acc, mode='lines+markers', name='Pure Acc', line=dict(color='green', width=3)), secondary_y=True)
                fig_global.add_trace(go.Scatter(x=steps_sorted, y=avg_forced_acc, mode='lines+markers', name='Forced Acc', line=dict(color='#8e44ad', width=2, dash='dot')), secondary_y=True) 
                fig_global.add_trace(go.Scatter(x=steps_sorted, y=avg_fast_acc, mode='lines+markers', name='Fast Acc', line=dict(color='#16a085', width=2, dash='dash')), secondary_y=True)
                
                loss_axis_type = "log" if toggle_scale.value == '对数坐标 (Log)' else "linear"
                fig_global.update_layout(title="📈 全局平均优化趋势 (Loss与Acc)", height=450, hovermode="x unified", template="plotly_white", margin=dict(t=50, b=20, l=20, r=20))
                fig_global.update_yaxes(title_text=f"Loss ({loss_axis_type})", type=loss_axis_type, secondary_y=False)
                fig_global.update_yaxes(title_text="Accuracy", range=[0, 1.05], secondary_y=True)
                fig_global.show()
                
                fig_len = go.Figure()
                fig_len.add_trace(go.Scatter(x=steps_sorted, y=avg_gen_len, mode='lines+markers', fill='tozeroy', name='Avg Token Length', line=dict(color='#8e44ad', width=2)))
                fig_len.update_layout(title="📏 全局平均生成文本长度 (Tokens)", height=300, hovermode="x unified", template="plotly_white", margin=dict(t=50, b=20, l=20, r=20))
                fig_len.update_yaxes(title_text="Token Count")
                fig_len.show()

        def plot_metrics(sample_id):
            resps = all_data[sample_id]
            fig = make_subplots(rows=3, cols=3, subplot_titles=(
                "Pure Accuracy", "GT Loss", "LM Loss", 
                "Token Change Ratio", "KL Loss", "ROUGE-L",
                "Gen Text Length (Tokens)", "Forced Accuracy", "Fast Accuracy"
            ))
            colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
            
            for i, (uid, record) in enumerate(resps.items()):
                color = colors[i % len(colors)]
                steps = record['steps']
                
                step_nums = [s['step'] for s in steps]
                gt_losses = [s['metrics'].get('gt_loss', None) for s in steps]
                lm_losses = [s['metrics'].get('lm_loss', None) for s in steps]
                kl_losses = [s['metrics'].get('kl_loss', None) for s in steps]
                
                eval_steps = [s['step'] for s in steps if 'pure_acc' in s['metrics']]
                pure_accs = [s['metrics'].get('pure_acc') for s in steps if 'pure_acc' in s['metrics']]
                change_ratios = [s['metrics'].get('change_ratio') for s in steps if 'change_ratio' in s['metrics']]
                rouge_ls = [s['metrics'].get('rouge_L') for s in steps if 'rouge_L' in s['metrics']]
                
                len_steps = [s['step'] for s in steps if 'gen_text_len' in s['metrics']]
                gen_lens = [s['metrics'].get('gen_text_len') for s in steps if 'gen_text_len' in s['metrics']]
                forced_steps = [s['step'] for s in steps if 'forced_acc' in s['metrics']]
                forced_accs = [s['metrics'].get('forced_acc') for s in steps if 'forced_acc' in s['metrics']]
                fast_steps = [s['step'] for s in steps if 'fast_acc' in s['metrics']]
                fast_accs = [s['metrics'].get('fast_acc') for s in steps if 'fast_acc' in s['metrics']]
                
                if eval_steps: fig.add_trace(go.Scatter(x=eval_steps, y=pure_accs, mode='lines+markers', name=uid, legendgroup=uid, line=dict(color=color)), row=1, col=1)
                fig.add_trace(go.Scatter(x=step_nums, y=gt_losses, mode='lines', name=uid, legendgroup=uid, showlegend=False, line=dict(color=color)), row=1, col=2)
                fig.add_trace(go.Scatter(x=step_nums, y=lm_losses, mode='lines', name=uid, legendgroup=uid, showlegend=False, line=dict(color=color, dash='dot')), row=1, col=3)
                
                if eval_steps and change_ratios: fig.add_trace(go.Scatter(x=eval_steps, y=change_ratios, mode='lines+markers', name=uid, legendgroup=uid, showlegend=False, line=dict(color=color)), row=2, col=1)
                fig.add_trace(go.Scatter(x=step_nums, y=kl_losses, mode='lines', name=uid, legendgroup=uid, showlegend=False, line=dict(color=color)), row=2, col=2)
                if eval_steps and rouge_ls: fig.add_trace(go.Scatter(x=eval_steps, y=rouge_ls, mode='lines+markers', name=uid, legendgroup=uid, showlegend=False, line=dict(color=color)), row=2, col=3)
                
                if len_steps and gen_lens: fig.add_trace(go.Scatter(x=len_steps, y=gen_lens, mode='lines+markers', name=uid, legendgroup=uid, showlegend=False, line=dict(color=color, dash='dashdot')), row=3, col=1)
                if forced_steps and forced_accs: fig.add_trace(go.Scatter(x=forced_steps, y=forced_accs, mode='lines+markers', name=uid, legendgroup=uid, showlegend=False, line=dict(color=color)), row=3, col=2)
                if fast_steps and fast_accs: fig.add_trace(go.Scatter(x=fast_steps, y=fast_accs, mode='lines+markers', name=uid, legendgroup=uid, showlegend=False, line=dict(color=color)), row=3, col=3)

            fig.update_layout(height=850, title_text=f"问题 {sample_id} 变体演化对比", hovermode="x unified", template="plotly_white")
            loss_axis_type = "log" if toggle_scale.value == '对数坐标 (Log)' else "linear"
            fig.update_yaxes(type=loss_axis_type, row=1, col=2) 
            fig.update_yaxes(type=loss_axis_type, row=1, col=3) 
            fig.update_yaxes(type=loss_axis_type, row=2, col=2) 
            
            return fig

        def draw_local_plot():
            sid = sample_stats[dropdown_sample.value]
            with out_plots:
                clear_output(wait=True)
                fig = plot_metrics(sid)
                fig.show()

        # 🆕 更新文本展示逻辑，适配单选按钮
        def update_text_view(*args):
            uid = dropdown_uid.value
            step_idx = slider_step.value
            is_markdown = btn_markdown.value # 获取按钮的布尔值状态
            
            if not uid: return
            
            sid = sample_stats[dropdown_sample.value]
            record = all_data[sid][uid]
            target_step_data = next((s for s in record['steps'] if s['step'] == step_idx), None)
            
            with out_text:
                clear_output(wait=True)
                if target_step_data and 'sample_gen_text' in target_step_data['metrics']:
                    gen_text = target_step_data['metrics']['sample_gen_text']
                    gen_len = target_step_data['metrics'].get('gen_text_len', '未知')
                    
                    if not is_markdown: # 显示原始文本
                        html_content = f"""
                        <div style="padding: 15px; border: 1px solid #c8d6e5; border-radius: 8px; background-color: #f1f2f6;">
                            <h4 style="margin-top:0; color: #222f3e;">🚀 Step {step_idx} - 模型生成文本 (Raw) <span style="font-size:12px; color:#7f8c8d; font-weight:normal;">[{gen_len} Tokens]</span></h4>
                            <pre style="white-space: pre-wrap; font-family: 'Consolas', 'Monaco', monospace; font-size: 13px; line-height: 1.5; color: #341f97;">{gen_text}</pre>
                        </div>
                        """
                        display(HTML(html_content))
                    else: # Markdown 渲染
                        display(HTML(f"""
                        <div style="padding: 10px 15px; border: 1px solid #74b9ff; border-bottom: none; border-radius: 8px 8px 0 0; background-color: #eccc68;">
                            <h4 style="margin:0; color: #2d3436;">🚀 Step {step_idx} - 模型生成文本 (Markdown 渲染) <span style="font-size:12px; font-weight:normal;">[{gen_len} Tokens]</span></h4>
                        </div>
                        """))
                        display(Markdown(gen_text))
                else:
                    display(HTML(f"<div style='color:#e17055; padding:10px;'>ℹ️ Step {step_idx} 没有记录生成文本</div>"))

        # 🆕 添加按钮动态状态更新逻辑
        def on_markdown_toggle(change):
            if change['new']: # 变为 True
                btn_markdown.description = ' 📝 原始文本'
                btn_markdown.button_style = 'success'
            else: # 变为 False
                btn_markdown.description = ' 💡 渲染 Markdown'
                btn_markdown.button_style = 'info'
            update_text_view()

        def on_scale_change(change):
            if change['type'] == 'change' and change['name'] == 'value':
                draw_global_plot()
                if dropdown_sample.value: draw_local_plot()

        def on_sample_change(change):
            if change['type'] == 'change' and change['name'] == 'value':
                sid = sample_stats[change['new']]
                resps = all_data[sid]
                first_record = list(resps.values())[0]
                
                with out_problem:
                    clear_output(wait=True)
                    display(HTML(f"""
                    <div style="padding: 12px; border-left: 5px solid #0984e3; background: #e3f2fd; border-radius: 4px;">
                        <b style="color: #0984e3;">🎯 原始问题:</b> {first_record['problem']}<br><br>
                        <b style="color: #00b894;">✅ 标准答案:</b> {first_record['gt_text']}
                    </div>
                    """))
                    
                draw_local_plot()
                    
                dropdown_uid.options = list(resps.keys())
                if len(resps) > 0:
                    uid = list(resps.keys())[0]
                    dropdown_uid.value = uid
                    eval_steps = [s['step'] for s in resps[uid]['steps'] if 'sample_gen_text' in s['metrics']]
                    slider_step.options = eval_steps if eval_steps else [0]
                    update_text_view()

        toggle_scale.observe(on_scale_change)
        dropdown_sample.observe(on_sample_change)
        dropdown_uid.observe(lambda x: setattr(slider_step, 'options', [s['step'] for s in all_data[sample_stats[dropdown_sample.value]][dropdown_uid.value]['steps'] if 'sample_gen_text' in s['metrics']]) if x['name'] == 'value' else None)
        slider_step.observe(update_text_view, names='value')
        
        # 🆕 绑定新按钮的事件监听
        btn_markdown.observe(on_markdown_toggle, names='value')

        # ==========================================
        # D. 页面布局组装
        # ==========================================
        print("="*60)
        print("📈 第一部分：全局大盘趋势 (Global Average Trends)")
        print("="*60)
        display(toggle_scale)
        display(out_global_plot)
        draw_global_plot()
        
        print("\n" + "="*60)
        print("🔍 第二部分：细粒度个案诊断 (Sample-wise Diagnosis)")
        print("="*60)
        display(dropdown_sample)
        display(out_problem)
        display(out_plots)
        
        display(HTML("<hr><h3 style='color: #2d3436;'>🔍 潜变量时光机 (Time Machine)</h3>"))
        # 🆕 替换 HBox 中的 toggle 组件
        display(widgets.HBox([dropdown_uid, slider_step, btn_markdown]))
        display(out_text)

        if sample_stats:
            on_sample_change({'type': 'change', 'name': 'value', 'new': dropdown_sample.value})

btn_load.on_click(lambda b: run_analysis(file_dropdown.value))

⏳ 正在加载 Qwen Tokenizer...


📊 Latent Space Optimization 终极可视化分析引擎


Output()